In [1]:
import csv
from collections import Counter
from pathlib import Path

import pandas as pd

from textmining.hit_utils import HitProcessor
from textmining.synonym_utils import MultiSynFileReader

RUN_DIR = Path('../outputs/new_full_run_w_genes')
HITS = RUN_DIR / 'new_full_run_w_genes.hits'
TOP_N = 1000
CHUNKSIZE = 5_000_000

pd.set_option('display.max_rows', TOP_N)
pd.set_option('display.max_colwidth', 60)

In [2]:
# Stream the 14 GB file: only synonym_id (col 1) and synonym (col 5) are needed.
# Count (id, synonym) pairs so the display name can be picked afterwards without a second pass.
pair_counts = Counter()
reader = pd.read_csv(HITS, sep='\t', header=None, usecols=[1, 5], names=['synonym_id', 'synonym'],
                     dtype=str, quoting=csv.QUOTE_NONE, keep_default_na=False, chunksize=CHUNKSIZE)
for chunk in reader:
    pair_counts.update(chunk.groupby(['synonym_id', 'synonym'], sort=False).size().to_dict())

pairs = pd.Series(pair_counts, name='count').rename_axis(['synonym_id', 'synonym']).reset_index()
print(f'{pairs["count"].sum():,} hits, {pairs["synonym_id"].nunique():,} distinct synonym ids')

688,137,034 hits, 278,880 distinct synonym ids


In [3]:
# Aggregate per synonym_id; display name = most frequent surface string for that id.
by_id = pairs.groupby('synonym_id').agg(hits=('count', 'sum'), n_forms=('synonym', 'size'))
top_name = pairs.sort_values('count', ascending=False).drop_duplicates('synonym_id').set_index('synonym_id')['synonym']
by_id = by_id.join(top_name)
total_hits = by_id['hits'].sum()

# Entity type and synonym file come from the file index of the id; cheap enough for all ids.
synfile_map = HitProcessor.parse_synfile_map(RUN_DIR / 'synfile.map')
type_map = HitProcessor.parse_synfile_type_map(RUN_DIR / 'synfile_type.map')
file_ids = by_id.index.str.split(':').str[0]
by_id['synfile'] = file_ids.map(lambda f: synfile_map[f].stem)
by_id['type'] = file_ids.map(lambda f: type_map[synfile_map[f].name][0].name)

def finalize(df):
    """Resolve entity ids (only for the selected rows) and order columns."""
    df = df.copy()
    with MultiSynFileReader(low_memory=True) as syn_reader:
        df['entity_id'] = [syn_reader.extract_id(str(synfile_map[sid.split(':')[0]]), int(sid.split(':')[1]))
                           for sid in df.index]
    df['pct_hits'] = (100 * df['hits'] / total_hits).round(3)
    df = df[['synonym', 'type', 'entity_id', 'synfile', 'hits', 'pct_hits', 'n_forms']].reset_index()
    df.index += 1
    return df

top = finalize(by_id.nlargest(TOP_N, 'hits'))
#top

In [4]:
# Top N per entity type, one TSV each (pct_hits stays relative to all hits in the run).
OUT_DIR = Path('.')
for hit_type, group in by_id.groupby('type'):
    out = OUT_DIR / f'top_synonyms_{hit_type}.tsv'
    finalize(group.nlargest(TOP_N, 'hits')).to_csv(out, sep='\t', index_label='rank')
    print(f'{hit_type:<20} {group["hits"].sum():>14,} hits  {len(group):>9,} ids  -> {out}')

BIOLOGICAL_PROCESS       26,706,820 hits     14,711 ids  -> top_synonyms_BIOLOGICAL_PROCESS.tsv
CELL                      4,895,285 hits      1,716 ids  -> top_synonyms_CELL.tsv
DISEASE                  86,157,676 hits     21,406 ids  -> top_synonyms_DISEASE.tsv
GENE                    459,493,878 hits    161,105 ids  -> top_synonyms_GENE.tsv
MIR                      27,361,293 hits          7 ids  -> top_synonyms_MIR.tsv
PATHWAY                  11,701,536 hits      1,034 ids  -> top_synonyms_PATHWAY.tsv
TAXON                    40,427,283 hits     74,973 ids  -> top_synonyms_TAXON.tsv
TISSUE                   31,393,263 hits      3,928 ids  -> top_synonyms_TISSUE.tsv
